# OPTIMA — Kaggle: Heavy Hugging Face LLM Enrichment Experiment

Compares Hugging Face causal LMs as the **enrichment** stage of the Optima pipeline:

```
GitHub base.json (frozen snapshot)
        |
        v
  LLM enrichment (gated: load -> trivial -> 1 function -> 3 functions -> full 96)
        |
        v
  embeddings (optima.rag, unchanged) -> FAISS indexes
        |
        v
  retrieval (unchanged) -> Recall@K / MRR evaluation
```

**How to use this notebook**

1. Edit the `CONFIG` cell: `BASE_JSON` (a local Kaggle-input path or a URL) and `MODEL_NAME`
   (one Hugging Face model per run). `BASE_JSON` picks the dataset/test-suite, `MODEL_NAME`
   picks the enrichment model -- the two are completely independent, and every output path is
   keyed by both (`optima_outputs/runs/<dataset>/...`), so different datasets and different
   models never overwrite each other's results.
2. *Run All*. Each model must pass GATES 1-4 before the full 96-function run starts.
3. For a run that may take hours, use **Save & Run All (Commit)** so the checkpoint
   survives a lost session; resume with `RESUME_INPUT_DIR` pointed at the previous
   version's output dataset.
4. The analyzer / libclang are **not** used here: this notebook consumes an existing
   `base.json` (produced locally by `optima analyze <test-suite-dir>`, one per test suite
   under `optima_outputs/base/<test-suite>/base.json`) and never regenerates it.
5. Every enrichment model is evaluated against the exact same base.json snapshot and
   the exact same frozen benchmark, so `summaries/comparison.csv` is a fair comparison.
6. **Multi-GPU:** on a session with two or more GPUs (e.g. Kaggle's T4 x2), a model
   that does not fit on one GPU is automatically sharded across all of them via
   accelerate (`device_map="auto"` + an explicit `max_memory`), not left unused --
   `check_fit()` decides this at runtime and GATE 1 prints the resulting
   `model.hf_device_map` so you can confirm both GPUs are in use for a 30B-class
   model. A model that fits on one GPU stays on one GPU even in a multi-GPU
   session. On a single-GPU session everything degrades to plain single-GPU loading.


## Cell 1 — Configuration

In [ ]:
import os
from pathlib import Path

# ---- Optima repository (this project) ----
OPTIMA_REPO_URL = "https://github.com/I1gorr/optima_python.git"
OPTIMA_REF = "main"  # pin to a commit SHA for full reproducibility

# ---- Filesystem roots ----
OPTIMA_DIR = Path("/kaggle/working/optima-python")
OUTPUT_ROOT = Path("/kaggle/working/optima_outputs")

# =========================
# EXPERIMENT CONFIGURATION
# =========================
# BASE_JSON selects the dataset/test-suite; MODEL_NAME selects the
# enrichment model. These are completely independent -- change one without
# touching the other, and every output path is keyed by both, so e.g.
# chess-engine+DeepSeek-7B and chess-engine+Qwen2.5-14B (or
# project-a+DeepSeek-7B) each land in their own directory and never
# overwrite one another (see snapshot.load_base_snapshot / Cell 8).

# openssl base.json: located by searching, never by hardcoding a mount
# path -- Kaggle has used at least two different conventions for this
# dataset across sessions (/kaggle/input/openssl/base.json vs.
# /kaggle/input/datasets/kushagra123rr/openssl/...). If the dataset isn't
# attached as an Input at all, it's downloaded directly via kagglehub
# (preinstalled on Kaggle, auto-authenticated with this notebook's own
# credentials -- no kaggle.json/API key needed).
_OPENSSL_DATASET_SLUG = "kushagra123rr/openssl"


def _find_base_json(root):
    if not os.path.isdir(root):
        return None
    for _dirpath, _dirnames, _filenames in os.walk(root):
        if "base.json" in _filenames:
            return os.path.join(_dirpath, "base.json")
    return None


_base_json_path = _find_base_json("/kaggle/input")

if _base_json_path is None:
    try:
        import kagglehub as _kagglehub
    except ImportError:
        import subprocess as _subprocess
        _subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub as _kagglehub

    _openssl_dir = _kagglehub.dataset_download(_OPENSSL_DATASET_SLUG)
    print(f"kagglehub downloaded {_OPENSSL_DATASET_SLUG!r} to {_openssl_dir}")
    _base_json_path = _find_base_json(_openssl_dir)

if _base_json_path is None:
    raise FileNotFoundError(
        "No base.json found under /kaggle/input or the kagglehub download of "
        f"{_OPENSSL_DATASET_SLUG!r}."
    )

BASE_JSON = _base_json_path
print(f"Resolved BASE_JSON = {BASE_JSON!r}")

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

QUANTIZATION = "nf4"                      # "nf4" (needs bitsandbytes) or "fp16"
MAX_INPUT_TOKENS = 1500                   # per-function prompt budget; the reduction ladder in
                                           # build_bounded_messages() trims further if still over this

# ---- Dataset identity (usually leave this alone) ----
# The dataset/test-suite name is normally read from BASE_JSON's own content
# (project.test_suite, falling back to project.name) -- stable regardless of
# where the file happens to be mounted or how it was renamed. Set DATASET to
# override that detection (e.g. two different base.json revisions for the
# same test suite that you deliberately want to compare as one dataset).
DATASET = None                            # None = auto-detect from BASE_JSON's own content
BASE_JSON_EXPECTED_SHA256 = None          # set to a sha256 hex digest to pin an exact snapshot
EXPECTED_FUNCTION_COUNT = None            # openssl base.json function count is unknown up front; set an int to pin it

# ---- Benchmark (generated once from base.json only; never regenerated per-model) ----
BENCHMARK_JSON_URL = None              # None = generate from base.json; or a curated queries.json URL
NUM_QUERIES = 20
BENCHMARK_SEED = 42

# ---- Model placement: check_fit()/load_model_safe() always shard this one
# model instance across every visible GPU (e.g. both T4s) rather than
# placing it on a single GPU, even when it would fit alone on one. One
# notebook run enriches with exactly one model -- there is no registry,
# queue, or model-selection logic.

# ---- Generation ----
PROMPT_VARIANT = "colab_v2_no_module_ir"  # excludes AST/CFG/module-level LLVM IR by default,
                                           # keeping each function's prompt in the ~500-1500 token range
MAX_NEW_TOKENS = None                     # None = no artificial output cap; the model generates
                                           # until it emits EOS or exhausts its own context window
                                           # (see models.max_new_tokens_for_context). This is a pure
                                           # model-comparison run: set an int here only if you want to
                                           # deliberately compare models under a fixed output budget.
DO_SAMPLE = False                         # greedy decoding for deterministic, reproducible output
TEMPERATURE = None
TOP_P = None
RETRIES = 2

# ---- Full-run safety ----
# There is no consecutive-failure/success-rate circuit breaker: the run never
# stops because of a model's OUTPUT (invalid_json, insufficient context, a
# schema-incomplete response, ...) -- every one of the 96 functions is recorded
# and the run continues. Only a genuine infrastructure failure (an unrecovered
# CUDA OOM, a model/tokenizer that fails to load, ...) stops it, by raising.
MATERIALIZE_EVERY = 25                     # write enriched_<slug>.json every 25 functions instead of
                                           # every single one -- openssl's base.json is far larger than
                                           # the 96-function dataset this notebook was tuned against, and
                                           # every materialize() call still serializes+writes the whole
                                           # artifact once (run_full_enrichment now reuses one in-memory
                                           # copy across calls instead of reloading/deep-copying base.json
                                           # per call, but the per-call json.dumps+write is still real
                                           # work). The checkpoint (enriched/<slug>.checkpoint.jsonl), not
                                           # this file, is what makes a run resumable, so lowering this to
                                           # 1 never loses progress -- it only trades I/O for staleness of
                                           # the human-readable enhanced_<slug>.json between writes.
SMOKE_FUNCTION_IDS = None                 # e.g. ["path.cpp::func::12"] to pin the smoke-test functions
RUN_FULL_ENRICHMENT = True

# ---- Embedding / retrieval / evaluation (runs only after the GPU is free of any LLM) ----
RUN_EVALUATION = True
EMBEDDING_MODELS = ["bge-small"]
REPRESENTATION_MODES = ["hybrid", "semantic"]
K = 10

# ---- Resume (attach a previous version's /kaggle/working/optima_outputs as a dataset) ----
RESUME_INPUT_DIR = None   # e.g. Path("/kaggle/input/optima-outputs-v1/optima_outputs")

# ---- Environment / cleanup ----
INSTALL_BITSANDBYTES_IF_MISSING = True
DELETE_MODEL_CACHE_AFTER_UNLOAD = True

HANDLE = None  # the only variable ever allowed to hold a loaded model

# The exact slug ModelSpec will use for this MODEL_NAME (matches
# optima.rag.embedding_simple._slug, which models.spec_from_model_id() calls
# later); computed here, before any optima_kaggle import, only so the run
# directory's manifest can record which model this run is for.
import re as _re
MODEL_SLUG = _re.sub(r"[^a-zA-Z0-9._-]+", "-", MODEL_NAME.strip()).strip("-").lower()

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# CUDA_VISIBLE_DEVICES is deliberately left unset: both GPUs (e.g. a Kaggle
# T4 x2 session) must stay visible so a model that needs sharding can use
# them both. torch.cuda.device_count(), read in the diagnostics cell below,
# is what the rest of this notebook branches on -- never an env var.
os.environ.setdefault("HF_HOME", "/tmp/hf_home")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OPTIMA_EMBEDDING_DEVICE", "cuda")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "warning")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Configuration loaded. BASE_JSON={BASE_JSON!r}")
print(f"MODEL_NAME={MODEL_NAME!r} (slug={MODEL_SLUG!r}), quantization={QUANTIZATION!r}")


## Cell 2 — Kaggle environment diagnostics (pre-clone)

In [ ]:
import platform
import subprocess

print(f"Python version: {platform.python_version()}")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout)
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")


## Cell 3 — Clone / checkout Optima Python (sparse, shallow; no pip install)

In [ ]:
import subprocess
import sys


def _run(cmd, **kwargs):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result


if not (OPTIMA_DIR / ".git").is_dir():
    if OPTIMA_DIR.exists():
        raise RuntimeError(f"{OPTIMA_DIR} exists but is not a git repository; remove it first.")
    _run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
          OPTIMA_REPO_URL, str(OPTIMA_DIR)])
    _run(["git", "sparse-checkout", "set", "--cone", "optima", "colab", "optima_kaggle"], cwd=OPTIMA_DIR)
    _run(["git", "fetch", "--depth", "1", "origin", OPTIMA_REF], cwd=OPTIMA_DIR)
    _run(["git", "checkout", "FETCH_HEAD"], cwd=OPTIMA_DIR)
else:
    status = _run(["git", "status", "--porcelain"], cwd=OPTIMA_DIR)
    if status.stdout.strip():
        raise RuntimeError(f"{OPTIMA_DIR} has uncommitted changes; refusing to touch it.")
    _run(["git", "fetch", "--depth", "1", "origin", OPTIMA_REF], cwd=OPTIMA_DIR)
    _run(["git", "checkout", "FETCH_HEAD"], cwd=OPTIMA_DIR)

# A fresh kernel never has stale modules, but a re-run of this cell might.
for name in list(sys.modules):
    if name == "optima" or name.startswith(("optima.", "optima_kaggle", "colab")):
        del sys.modules[name]
if str(OPTIMA_DIR) not in sys.path:
    sys.path.insert(0, str(OPTIMA_DIR))

OPTIMA_COMMIT = _run(["git", "rev-parse", "HEAD"], cwd=OPTIMA_DIR).stdout.strip()
print(f"Optima checked out at {OPTIMA_DIR}, commit {OPTIMA_COMMIT}")


## Cell 4 — Install/check Python dependencies (never touches torch or clang)

In [ ]:
from optima_kaggle import environment as env

DEPS = env.ensure_python_deps(allow_install=True)


## Cell 5 — GPU/CUDA/internet diagnostics

In [ ]:
ENV_INFO = env.diagnose(hf_home=os.environ.get("HF_HOME"), working_dir="/kaggle/working")
NUM_GPUS = ENV_INFO["num_gpus"]
print(f"NUM_GPUS = {NUM_GPUS} (multi_gpu_capable={ENV_INFO['multi_gpu_capable']})")
env.gpu_report("session start")


## Cell 6 — bitsandbytes compatibility probe (fails loud, never silently downgrades)

In [ ]:
BNB = env.probe_bitsandbytes(INSTALL_BITSANDBYTES_IF_MISSING)
print(f"bitsandbytes: ok={BNB.ok}, stage={BNB.stage}, version={BNB.version}")
print(BNB.message)


## Cell 7 — Import Optima + optima_kaggle; verify the analyzer/libclang are not loaded

In [ ]:
import inspect
import sys as _sys

from colab import colab_pipeline
from optima.rag import embedding_simple
from optima_kaggle import enrichment, models, retrieval_eval, snapshot

assert "optima.analyzer" not in _sys.modules, "the analyzer must not be imported for this experiment"
assert "clang" not in _sys.modules and "clang.cindex" not in _sys.modules, "libclang must not be imported"

assert ".tmp" in inspect.getsource(colab_pipeline.save_json), (
    "colab_pipeline.save_json is not atomic on this checkout; push the local fix to "
    "colab/colab_pipeline.py on GitHub before running this notebook."
)
assert "OPTIMA_EMBEDDING_DEVICE" in inspect.getsource(embedding_simple), (
    "optima.rag.embedding_simple lacks OPTIMA_EMBEDDING_DEVICE support on this checkout; "
    "push the local fix on GitHub before running this notebook (otherwise embeddings run on CPU)."
)
print("Imports OK; analyzer/libclang not loaded; required upstream fixes are present.")


## Cell 8 — Resolve BASE_JSON (local path or URL) into a dataset-addressed snapshot (loaded exactly once per run)

In [ ]:
SNAP = snapshot.load_base_snapshot(
    BASE_JSON, OUTPUT_ROOT, expected_sha256=BASE_JSON_EXPECTED_SHA256, dataset=DATASET,
)

CTX = snapshot.RunContext.create(
    OUTPUT_ROOT, SNAP,
    config={
        "prompt_variant": PROMPT_VARIANT, "max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE,
        "temperature": TEMPERATURE, "top_p": TOP_P, "retries": RETRIES,
        "embedding_models": EMBEDDING_MODELS, "representation_modes": REPRESENTATION_MODES, "k": K,
    },
    env=ENV_INFO, optima_commit=OPTIMA_COMMIT, configured_models=[MODEL_SLUG],
)
print(f"Dataset: {SNAP.dataset}")
print(f"Run directory: {CTX.run_dir}")


## Cell 9 — Validate base.json and print the resolved experiment configuration

In [ ]:
import json as _json

BASE_REPORT = snapshot.validate_base_snapshot(SNAP.path, EXPECTED_FUNCTION_COUNT)

_base_data = _json.loads(SNAP.path.read_text())
_num_nodes = len(_base_data.get("graph", {}).get("function_nodes", []))
print()
print("EXPERIMENT CONFIGURATION")
print(f"  Dataset:            {SNAP.dataset}")
print(f"  Base JSON:          {SNAP.path}  (source: {SNAP.url})")
print(f"  Number of functions:{BASE_REPORT['functions']:>6}")
print(f"  Number of nodes:    {_num_nodes:>6}")
print(f"  Base JSON fingerprint (sha256): {SNAP.sha256}")
print(f"  Model:              {MODEL_NAME}  (slug={MODEL_SLUG})")
print(f"  Quantization:       {QUANTIZATION}")
print(f"  Output directory:   {CTX.run_dir}")


## Cell 10 — Restore resumable state from a previous run (no-op if RESUME_INPUT_DIR is None)

In [ ]:
snapshot.restore_resume_state(CTX, RESUME_INPUT_DIR)


## Cell 11 — Freeze the benchmark (generated once from base.json only; never from enriched JSON)

In [ ]:
BENCH = snapshot.freeze_benchmark(CTX, NUM_QUERIES, BENCHMARK_SEED, BENCHMARK_JSON_URL)


---
## Model stages (`MODEL_NAME`)

GATE 1 (load) -> GATE 2 (trivial generation) -> GATE 3 (one real function) ->
GATE 4 (three real functions) -> full 96-function run. A failed gate raises and
stops here; it does not fall through to the full run.

## Cell 12 — Resolve the model spec, generation settings, and pre-load VRAM fit estimate

In [ ]:
# One model, built directly from MODEL_NAME -- no registry, no queue.
# check_fit() (below) is the only feasibility gate: it decides single-GPU vs
# sharded-across-both-T4s placement for THIS model at runtime and prints a
# per-GPU fit table; there is no registry tier to consult either way.
SPEC = models.spec_from_model_id(
    MODEL_NAME, quantization=QUANTIZATION,
    max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
)
assert SPEC.slug == MODEL_SLUG  # sanity: matches the slug computed in the config cell
print(f"Model: {SPEC.model_id}  (slug={SPEC.slug}, quantization={SPEC.quantization}, "
      f"max_input_tokens={SPEC.max_input_tokens}, max_new_tokens={SPEC.max_new_tokens})")

GEN = enrichment.GenerationSettings(
    do_sample=DO_SAMPLE, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    retries=RETRIES, prompt_variant=PROMPT_VARIANT,
)
FIT = models.check_fit(SPEC, num_gpus=NUM_GPUS)


## Cell 13 — GATE 1: model loads onto the GPU

In [ ]:
try:
    env.gpu_report("before load", CTX.gpu_log_path)
    HANDLE = models.load_model_safe(SPEC, BNB)
    env.gpu_report("after load", CTX.gpu_log_path)
    # load_model_safe() already printed the full hf_device_map; this is a
    # one-line summary so a sharded 30B-class load is obvious at a glance.
    print(f"placement={HANDLE.load_report['gpu_placement']}, "
          f"gpus_used={HANDLE.load_report['gpu_count_used']}, "
          f"used_cpu_offload={HANDLE.load_report['used_cpu_offload']}")
    G1 = enrichment.gate1_load(CTX, HANDLE, GEN)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 14 — GATE 2: trivial generation succeeds

In [ ]:
if HANDLE is None:
    raise RuntimeError(
        "GATE 2 requires a successfully loaded model HANDLE. "
        "GATE 1 must complete successfully first."
    )
try:
    G2 = enrichment.gate2_trivial(CTX, HANDLE, GEN)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 15 — GATE 3: one real function from base.json is enriched

In [ ]:
if HANDLE is None:
    raise RuntimeError(
        "GATE 3 requires a successfully loaded model HANDLE. "
        "GATE 1 must complete successfully first."
    )
try:
    G3 = enrichment.gate3_one_function(CTX, HANDLE, GEN, SNAP, SMOKE_FUNCTION_IDS)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 16 — GATE 4: three real functions are enriched (including the largest bounded prompt)

In [ ]:
if HANDLE is None:
    raise RuntimeError(
        "GATE 4 requires a successfully loaded model HANDLE. "
        "GATE 1 must complete successfully first."
    )
try:
    # Load model -> enrich 3 functions -> verify enrichment fields ->
    # verify the checkpoint was written. Same MAX_NEW_TOKENS as the full run.
    G4 = enrichment.gate4_three_functions(CTX, HANDLE, GEN, SNAP, SMOKE_FUNCTION_IDS)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 17 — Full resumable 96-function enrichment (only runs if GATES 1-4 passed)

In [ ]:
FULL = None
try:
    if RUN_FULL_ENRICHMENT:
        if HANDLE is None:
            raise RuntimeError(
                "Full enrichment requires a successfully loaded model HANDLE. "
                "GATE 1 must complete successfully first."
            )
        FULL = enrichment.run_full_enrichment(
            CTX, HANDLE, GEN, SNAP, materialize_every=MATERIALIZE_EVERY,
        )
        _metrics = FULL["metrics"]
        print()
        print("ENRICHMENT COMPLETE")
        print()
        print(f"Model: {MODEL_NAME}")
        print(f"Total functions: {_metrics.get('functions_total')}")
        print(f"Enriched: {_metrics.get('functions_enriched')}")
        print(f"Skipped/resumed (already done before this run): {FULL.get('resumed_functions', 0)}")
        print(f"Failed: {_metrics.get('functions_failed')}")
        print()
        print(f"Output:\n{FULL['enriched_json']}")
        print()
        print(_metrics)
    else:
        print("RUN_FULL_ENRICHMENT is False; skipping the full run.")
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 18 — Unload the model and verify GPU memory is released

In [ ]:
models.unload_model(HANDLE, delete_cache=DELETE_MODEL_CACHE_AFTER_UNLOAD)
HANDLE = None
env.gpu_report("after unload", CTX.gpu_log_path)


---
## Embedding, retrieval, evaluation

Everything below runs with **no LLM on the GPU** and reuses the existing
`optima.rag` embedding/retrieval/evaluation implementation unchanged.

## Cell 20 — Discover which corpora passed enrichment; verify the GPU is free of any LLM

In [ ]:
assert HANDLE is None, "A model handle is still live; unload it before embedding/retrieval."
env.assert_gpu_clean(threshold_gib=0.3)
CORPORA = retrieval_eval.discover_passed_corpora(CTX)
print("Corpora to embed/evaluate:", CORPORA)


## Cell 21 — Build FAISS indexes for raw + every passed enrichment model

In [ ]:
INDEXES = None
if RUN_EVALUATION:
    INDEXES = retrieval_eval.build_indexes(CTX, CORPORA, EMBEDDING_MODELS, REPRESENTATION_MODES)


## Cell 22 — Retrieval smoke test (pipeline sanity, not a quality bar)

In [ ]:
if RUN_EVALUATION:
    from optima.rag.embedding_simple import embedding_alias
    retrieval_eval.retrieval_smoke(
        CTX, CORPORA, embedding_alias(EMBEDDING_MODELS[0]), REPRESENTATION_MODES[0], BENCH
    )


## Cell 23 — Full retrieval evaluation (Recall@K, MRR, latency) for every mode/alias/corpus

In [ ]:
MATRICES = None
if RUN_EVALUATION:
    MATRICES = retrieval_eval.evaluate_all(CTX, EMBEDDING_MODELS, REPRESENTATION_MODES, BENCH, K, CORPORA)


## Cell 24 — Cross-model comparison table (same base.json + benchmark for every row)

In [ ]:
COMPARISON = None
if RUN_EVALUATION:
    COMPARISON = retrieval_eval.build_comparison(CTX, MATRICES, CORPORA, BENCH)
    try:
        import pandas as pd
        display(pd.DataFrame(COMPARISON["rows"]).sort_values(
            ["representation_mode", "embedding_alias", "mrr"], ascending=[True, True, False]
        ))
    except ImportError:
        for row in COMPARISON["rows"]:
            print(row)

    def _fmt(value):
        return f"{value:.4f}" if isinstance(value, (int, float)) else str(value)

    print()
    print("EVALUATION SUMMARY")
    print(f"Model: {MODEL_NAME}")
    for row in COMPARISON["rows"]:
        if row["corpus"] == MODEL_SLUG:
            print(f"  [{row['representation_mode']}/{row['embedding_alias']}] "
                  f"Recall@5={_fmt(row['recall_at_5'])}  MRR={_fmt(row['mrr'])}  "
                  f"Recall@10={_fmt(row['recall_at_10'])}")
    print(f"Full results: {CTX.summaries_dir / 'comparison.csv'}")


## Cell 25 — Final artifact report (raises if any configured model did not pass)

In [ ]:
snapshot.final_report(CTX)
